# Fase 2 — Experiment Tracking con MLflow

## Objetivo de este notebook

En la Fase 1.3 comparamos 10 modelos de clasificación "a mano", guardando los resultados en un DataFrame dentro del notebook de EDA. Eso funciona para explorar, pero tiene un problema para lo que sigue del proyecto: si cerramos el notebook o lo volvemos a correr, perdemos el detalle de cada corrida (qué hiperparámetros se usaron, qué métricas dio exactamente, qué versión del modelo fue) — y no hay forma fácil de comparar experimentos entre sí ni de saber cuál modelo entrenado es "el bueno" para pasar a producción.

**MLflow** resuelve esto: es una herramienta de *experiment tracking* que registra automáticamente, para cada entrenamiento (cada "run"):
- los **parámetros** usados (hiperparámetros del modelo, semillas, etc.),
- las **métricas** obtenidas (recall, precision, f1, AUC-PR),
- **artefactos** (el modelo entrenado, gráficos, archivos),
- y metadatos (cuándo se corrió, con qué código).

Todo esto queda guardado localmente en una carpeta `mlruns/` (que no versionamos en git — se regenera al correr el código, igual que la data), y se puede explorar después con una interfaz visual (`mlflow ui`) para comparar corridas entre sí.

### Qué vamos a hacer en este notebook
1. Configurar MLflow y crear un *experimento* (un espacio con nombre donde se agrupan los runs relacionados).
2. Cargar los datos ya preparados (mismo split y preprocesamiento del EDA).
3. Entrenar y loguear a MLflow los modelos que ya identificamos como más prometedores en la Fase 1.3 (empezando por Hist Gradient Boosting, el elegido).
4. Comparar corridas desde la interfaz de MLflow.

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    recall_score, precision_score, f1_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_sample_weight

import mlflow
import mlflow.sklearn

import optuna

## Paso 1: Configurar MLflow y crear el experimento

Antes de entrenar nada, hay que decirle a MLflow dos cosas:

1. **Dónde guardar los runs** (`tracking_uri`): por defecto MLflow los guarda en una carpeta `mlruns/` en el directorio donde se ejecuta el código. Como este notebook vive en `notebooks/`, si no le decimos nada, la carpeta `mlruns/` quedaría *dentro* de `notebooks/` — para mantener la misma estructura ordenada del resto del proyecto (igual que `../data/raw/...`), la apuntamos explícitamente a la raíz del proyecto.
2. **En qué experimento agrupar los runs** (`experiment_name`): un experimento es como una carpeta con nombre que agrupa runs relacionados — todos los runs de comparación de modelos para este dataset van a ir bajo el mismo experimento, para poder compararlos entre sí en la interfaz de MLflow.

In [2]:
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("mantenimiento-predictivo-ai4i2020")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento activo:", mlflow.get_experiment_by_name("mantenimiento-predictivo-ai4i2020"))

2026/08/31 18:46:12 INFO mlflow.tracking.fluent: Experiment with name 'mantenimiento-predictivo-ai4i2020' does not exist. Creating a new experiment.


Tracking URI: file:../mlruns
Experimento activo: <Experiment: artifact_location='file:///c:/Users/marin/Desktop/APUN/aprendizaje-automatico-en-la-nube/notebooks/../mlruns/662614544400800058', creation_time=1788219972624, experiment_id='662614544400800058', last_update_time=1788219972624, lifecycle_stage='active', name='mantenimiento-predictivo-ai4i2020', tags={}>


**Análisis:** el `tracking_uri` quedó apuntando correctamente a `file:../mlruns` — como el notebook vive en `notebooks/`, esa ruta relativa sube un nivel y crea la carpeta `mlruns/` en la raíz del proyecto (se confirma en el `artifact_location` de la salida: `.../aprendizaje-automatico-en-la-nube/notebooks/../mlruns/...`, que resuelve a la raíz).

Como el experimento `mantenimiento-predictivo-ai4i2020` no existía todavía, MLflow lo creó automáticamente (se ve en el log `INFO mlflow.tracking.fluent: Experiment with name ... does not exist. Creating a new experiment`). De aquí en adelante, cualquier run que loguemos va a quedar agrupado bajo este mismo experimento, listo para comparar en la interfaz de MLflow.

## Paso 2: Cargar los datos y recrear el split

Para que este notebook sea autosuficiente (no dependa de tener abierto el notebook de EDA), repetimos aquí la carga del dataset y el mismo split que usamos en la Fase 1.3 — **mismo `random_state=42` y mismo `stratify=y`**, para que el train/test sea idéntico y los resultados sean comparables entre notebooks.

Recordemos: excluimos de `X` las 5 banderas de tipo de falla (TWF, HDF, PWF, OSF, RNF) porque serían "fugas de información" (data leakage) — esas banderas básicamente ya delatan la falla. También excluimos `UID` y `Product ID` por ser identificadores, no variables predictivas.

In [5]:
df = pd.read_csv("../data/raw/ai4i2020.csv", encoding="utf-8-sig")

X = df.drop(columns=["UDI", "Product ID", "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"])
y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Proporcion de fallas en train:", y_train.mean().round(4))
print("Proporcion de fallas en test:", y_test.mean().round(4))

X_train: (8000, 6) | X_test: (2000, 6)
Proporcion de fallas en train: 0.0339
Proporcion de fallas en test: 0.034


**Análisis:** el split quedó con 8.000 observaciones en train y 2.000 en test (80/20, como en la Fase 1.3), y las 6 columnas restantes en `X` (`Type` + las 5 variables numéricas) tras excluir identificadores y banderas de falla. La proporción de fallas se mantuvo prácticamente igual en ambos conjuntos (3.39% en train, 3.40% en test) gracias al `stratify=y` — esto confirma que el desbalance de clases no se distorsionó al partir los datos, así que la comparación de modelos que hagamos aquí será consistente con la de la Fase 1.3.

## Paso 3: Preprocesamiento

Igual que en la Fase 1.3, armamos un `ColumnTransformer` que:
- Escala las 5 variables numéricas con `StandardScaler` (media 0, desviación 1) — importante para modelos sensibles a la escala.
- Convierte `Type` (L/M/H) a variables dummy con `OneHotEncoder(drop="first")` — se elimina la primera categoría para evitar redundancia (multicolinealidad).

In [6]:
variables_numericas = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
variables_categoricas = ["Type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), variables_numericas),
        ("cat", OneHotEncoder(drop="first"), variables_categoricas),
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Air temperature [K]',
                                  'Process temperature [K]',
                                  'Rotational speed [rpm]', 'Torque [Nm]',
                                  'Tool wear [min]']),
                                ('cat', OneHotEncoder(drop='first'), ['Type'])])


**Análisis:** el `ColumnTransformer` quedó configurado igual que en la Fase 1.3 — escala las 5 variables numéricas y convierte `Type` en variables dummy (elimina la categoría base para evitar redundancia). Con esto ya tenemos todo listo para entrenar y empezar a trackear con MLflow.

## Paso 4: Entrenar el modelo elegido (Hist Gradient Boosting) con tracking en MLflow

En la Fase 1.3 identificamos **Hist Gradient Boosting** como el modelo más robusto entre los 10 comparados. Ahora lo entrenamos de nuevo, pero esta vez envolviendo todo dentro de un **run de MLflow** (`mlflow.start_run()`), para que quede registrado automáticamente:

- **Parámetros** (`mlflow.log_param`): qué modelo es, cómo se manejó el desbalance de clases, la semilla usada.
- **Métricas** (`mlflow.log_metric`): Recall, Precision, F1 y AUC-PR de la clase "falla" — las mismas que usamos en la Fase 1.3, por consistencia.
- **El modelo entrenado como artefacto** (`mlflow.sklearn.log_model`): queda guardado el pipeline completo (preprocesamiento + modelo), listo para poder cargarlo después sin reentrenar.

Mantenemos el mismo manejo de desbalance que ya justificamos en la Fase 1.3: como `HistGradientBoostingClassifier` no soporta `class_weight` nativo, usamos `sample_weight` calculado con `compute_sample_weight`.

In [7]:
with mlflow.start_run(run_name="hist_gradient_boosting_baseline"):
    modelo = HistGradientBoostingClassifier(random_state=42)
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", modelo)])

    sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)
    pipeline.fit(X_train, y_train, classifier__sample_weight=sample_weight_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test, y_proba)

    mlflow.log_param("modelo", "HistGradientBoostingClassifier")
    mlflow.log_param("manejo_desbalance", "sample_weight (balanced)")
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("recall_falla", recall)
    mlflow.log_metric("precision_falla", precision)
    mlflow.log_metric("f1_falla", f1)
    mlflow.log_metric("auc_pr", auc_pr)

    mlflow.sklearn.log_model(pipeline, "modelo")

    print("Run ID:", mlflow.active_run().info.run_id)
    print(f"Recall: {recall:.4f} | Precision: {precision:.4f} | F1: {f1:.4f} | AUC-PR: {auc_pr:.4f}")

2026/08/31 18:51:04 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/08/31 18:51:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run ID: f57af4cd3b7c43d3b8d4bf38e96661fb
Recall: 0.7941 | Precision: 0.7297 | F1: 0.7606 | AUC-PR: 0.8343


**Análisis:** el primer run quedó registrado en MLflow (`Run ID: f57af4cd3b7c43d3b8d4bf38e96661fb`) con Recall 0.7941, Precision 0.7297, F1 0.7606 y AUC-PR 0.8343.

Comparando con la meta declarada en la Sección 1.1 (Recall ≥ 0.80 en la clase de falla), quedamos **muy cerca pero por debajo** (0.7941 vs. 0.80). Esto no es un problema — es justo la motivación para las tareas pendientes de esta fase: ajustar el umbral de decisión (por defecto el modelo predice con corte en 0.5, pero podemos bajarlo para priorizar más el Recall a costa de algo de Precision), tunear hiperparámetros con Optuna, y probar SMOTE. Ahora que tenemos este primer run como punto de referencia ("baseline") registrado en MLflow, cualquier mejora que hagamos después la podemos comparar objetivamente contra este número.

**Análisis:** confirmado visualmente en la interfaz de MLflow — el run `hist_gradient_boosting_baseline` quedó registrado dentro del experimento `mantenimiento-predictivo-ai4i2020`, con el modelo (pipeline completo de preprocesamiento + clasificador) guardado como artefacto. A partir de aquí, cada vez que entrenemos una variante (otro modelo, otros hiperparámetros, con SMOTE, etc.) va a aparecer como una fila más en esta misma tabla, lo que nos permite comparar experimentos de forma ordenada y objetiva — justo el problema que teníamos en la Fase 1.3 al comparar los 10 modelos "a mano" en un DataFrame que se perdía al cerrar el notebook.

## Paso 5: Validar con cross-validation

Hasta ahora evaluamos el modelo con un único train/test split (80/20) — eso da una sola medición que puede depender un poco de qué observaciones cayeron por azar en el test. La validación cruzada entrena y evalúa el modelo varias veces con distintas particiones, dándonos un promedio y una desviación estándar por métrica: una estimación más robusta de qué tan bien generaliza el modelo.

Usamos `StratifiedKFold` con 5 folds (mantiene la proporción de fallas en cada partición) sobre `X_train`/`y_train`, y logueamos el resultado como un nuevo run en MLflow, para poder compararlo contra el run del split simple.

In [10]:
with mlflow.start_run(run_name="hist_gradient_boosting_cv5"):
    modelo = HistGradientBoostingClassifier(random_state=42)
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", modelo)])

    sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    resultados_cv = cross_validate(
    pipeline, X_train, y_train,
    cv=cv,
    scoring=["recall", "precision", "f1"],
    params={"classifier__sample_weight": sample_weight_train},
)

    recall_mean, recall_std = resultados_cv["test_recall"].mean(), resultados_cv["test_recall"].std()
    precision_mean, precision_std = resultados_cv["test_precision"].mean(), resultados_cv["test_precision"].std()
    f1_mean, f1_std = resultados_cv["test_f1"].mean(), resultados_cv["test_f1"].std()

    mlflow.log_param("modelo", "HistGradientBoostingClassifier")
    mlflow.log_param("manejo_desbalance", "sample_weight (balanced)")
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("recall_falla_mean", recall_mean)
    mlflow.log_metric("recall_falla_std", recall_std)
    mlflow.log_metric("precision_falla_mean", precision_mean)
    mlflow.log_metric("precision_falla_std", precision_std)
    mlflow.log_metric("f1_falla_mean", f1_mean)
    mlflow.log_metric("f1_falla_std", f1_std)

    print(f"Recall: {recall_mean:.4f} +/- {recall_std:.4f}")
    print(f"Precision: {precision_mean:.4f} +/- {precision_std:.4f}")
    print(f"F1: {f1_mean:.4f} +/- {f1_std:.4f}")

Recall: 0.7712 +/- 0.0449
Precision: 0.6710 +/- 0.0557
F1: 0.7164 +/- 0.0433


**Análisis:** con cross-validation de 5 folds, Hist Gradient Boosting da Recall 0.7712 (± 0.0449), Precision 0.6710 (± 0.0557) y F1 0.7164 (± 0.0433).

Comparado con el run del split simple (Recall 0.7941, Precision 0.7297, F1 0.7606), el promedio de cross-validation queda un poco más bajo en las tres métricas. Esto es esperable y es justo el valor de hacer cross-validation: el split original nos dio una medición algo optimista por cómo cayeron los datos en ese test particular, mientras que el promedio de 5 folds es una estimación más realista y estable de cómo se comportaría el modelo con datos nuevos. La desviación estándar (±0.04-0.06) también nos dice que hay variabilidad razonable entre folds — no es un modelo perfectamente estable, algo a tener en cuenta antes de prometer un Recall exacto en producción.

## Paso 6: Loguear un segundo modelo para comparar (Random Forest)

Random Forest fue otro de los modelos con buen desempeño en la comparación de 10 modelos de la Fase 1.3. A diferencia de Hist Gradient Boosting, Random Forest **sí soporta `class_weight="balanced"` nativamente** (no necesita `sample_weight` por separado). Lo entrenamos igual, con cross-validation de 5 folds, y lo logueamos como otro run dentro del mismo experimento para poder comparar ambos modelos lado a lado en la interfaz de MLflow.

In [12]:
with mlflow.start_run(run_name="random_forest_cv5"):
    modelo = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", modelo)])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    resultados_cv = cross_validate(
        pipeline, X_train, y_train,
        cv=cv,
        scoring=["recall", "precision", "f1"],
    )

    recall_mean, recall_std = resultados_cv["test_recall"].mean(), resultados_cv["test_recall"].std()
    precision_mean, precision_std = resultados_cv["test_precision"].mean(), resultados_cv["test_precision"].std()
    f1_mean, f1_std = resultados_cv["test_f1"].mean(), resultados_cv["test_f1"].std()

    mlflow.log_param("modelo", "RandomForestClassifier")
    mlflow.log_param("manejo_desbalance", "class_weight balanced")
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("recall_falla_mean", recall_mean)
    mlflow.log_metric("recall_falla_std", recall_std)
    mlflow.log_metric("precision_falla_mean", precision_mean)
    mlflow.log_metric("precision_falla_std", precision_std)
    mlflow.log_metric("f1_falla_mean", f1_mean)
    mlflow.log_metric("f1_falla_std", f1_std)

    print(f"Recall: {recall_mean:.4f} +/- {recall_std:.4f}")
    print(f"Precision: {precision_mean:.4f} +/- {precision_std:.4f}")
    print(f"F1: {f1_mean:.4f} +/- {f1_std:.4f}")

Recall: 0.6643 +/- 0.0196
Precision: 0.7282 +/- 0.0624
F1: 0.6934 +/- 0.0313


**Análisis:** con cross-validation, Random Forest da Recall 0.6643 (± 0.0196), Precision 0.7282 (± 0.0624) y F1 0.6934 (± 0.0313).

Comparando ambos modelos con la misma metodología (CV de 5 folds):

| Modelo | Recall | Precision | F1 |
|---|---|---|---|
| Hist Gradient Boosting | 0.7712 ± 0.045 | 0.6710 ± 0.056 | 0.7164 ± 0.043 |
| Random Forest | 0.6643 ± 0.020 | 0.7282 ± 0.062 | 0.6934 ± 0.031 |

Random Forest tiene mejor Precision, pero **peor Recall** — y recordemos que Recall es la métrica prioritaria según la Sección 1.1 (no detectar una falla real es el error más caro). Esto confirma, ahora con cross-validation y no solo con un split, la elección que ya habíamos hecho en la Fase 1.3: **Hist Gradient Boosting sigue siendo el modelo más adecuado** para este problema, porque prioriza mejor el tipo de error que más nos interesa evitar.